# Synthetic Data Generation: Quality, Utility, and Privacy Diagnostics

**Purpose.** This notebook analyzes the bundled federated insurance dataset as a *synthetic* data artifact: per-client quality, cross-country distribution shift, inter-feature structure, train-on-one/test-on-another utility, and nearest-neighbor privacy diagnostics. It does **not** claim access to a real-world ground-truth source unless such data is explicitly provided.

## Learning objectives
- Assess tabular data quality per client with `insurance_fl.data.quality.DataQualityAnalyzer`.
- Compare country distributions and correlations to quantify heterogeneity (a key challenge for federated learning).
- Measure cross-country predictive utility with a train-on-A/test-on-B (TSTR-style) AUC matrix.
- Compute distance-to-closest-record (DCR) and duplicate rates as empirical privacy-risk indicators.
- Interpret results with explicit assumptions and limitations (synthetic ≠ anonymous).


## Background and math

### Data quality score
`DataQualityAnalyzer.overall_quality_score()` combines penalties for missingness, duplicates, and outlier rates into a 0–100 score. *Meaning here:* a compact summary of tabular hygiene per client; it is not a formal statistical validity test.

### Cross-country utility (TSTR-style matrix)
We train a classifier on country $A$ and evaluate AUC on country $B$:
$$
\mathrm{AUC}_{A\to B} = \Pr(s(x^+_B) > s(x^-_B) \mid f_A), \quad x_B \sim D_B.
$$
*Meaning here:* off-diagonal performance measures how transferable patterns are across countries; large gaps indicate non-IID/client heterogeneity.

### Distance to closest record (DCR)
For each query record $x$ in a held-out set $Q$ and a reference set $R$,
$$
\mathrm{DCR}(x;R)=\min_{r\in R} \|x-r\|_2.
$$
*Meaning here:* very small DCR/near-duplicates suggest memorization or insufficient perturbation. This is an empirical check, not a privacy guarantee.

### Correlation structure
We compare Pearson correlation matrices $C^{(c)}$ per country and visualize $|C^{(c)}-C^{(c')}|$.
*Meaning here:* differences indicate that relationships between features vary by client, motivating personalization/federated methods.


## Assumptions and limitations
- **Synthetic-only context:** bundled data in `data/processed/european_insurance_federated` is treated as generated/synthetic. Without a verified real source, we do not label comparisons as "real vs synthetic"; we report **client vs client** heterogeneity and **seed-A vs seed-B** stability when reproducible.
- DCR is computed on standardized numeric features and a light-mode sample; results depend on feature scaling and sample size.
- TSTR-style AUC uses `is_high_risk` as a bounded binary task with disjoint train/test splits; it is a utility proxy, not a full downstream benchmark.
- Synthetic data can leak information; low duplicate/DCR risk does not imply anonymity or differential privacy.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists():
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
env = nb.env_report()
print('NB_MODE:', nb.MODE)
env


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    n_rows: int = 2000 if nb.MODE == "light" else 5000
    n_pca: int = 1000 if nb.MODE == "light" else 4000
    n_nn: int = 800 if nb.MODE == "light" else 3000
    n_tsne: int = 500 if nb.MODE == "light" else 1000
cfg = Cfg(); cfg


## Data / inputs
Loaded from `nb.require_data()` (federated parquet partitions). In light mode each client is sampled to `cfg.n_rows` for speed; full mode uses up to all rows.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from insurance_fl.data.quality import DataQualityAnalyzer

data_root = nb.require_data()
print('data_root:', data_root)
clients = nb.load_clients(split="train", n_max=cfg.n_rows)
assert clients, 'No client datasets loaded'
for k, df in clients.items():
    print(k, df.shape)
sample_df = next(iter(clients.values()))
num_cols = [c for c in sample_df.select_dtypes(include=[np.number]).columns if c != 'segment_probabilities']
key_feats = [c for c in ["age", "annual_income", "claim_risk_score", "customer_lifetime_value"] if c in num_cols][:4]
print('key_feats:', key_feats)
key_feats


## Per-client quality analysis (baseline)

`DataQualityAnalyzer` is run per client. Scores and row counts are summarized in a table.


In [ ]:
rows = []
per_quality = {}
for cid, df in clients.items():
    aq = DataQualityAnalyzer()
    aq.analyze(df)
    q = float(aq.overall_quality_score())
    per_quality[cid] = {"analyzer": aq, "score": q}
    assert 0.0 <= q <= 100.0, f'quality score out of range for {cid}: {q}'
    rows.append(
        {
            "client": str(cid),
            "country": str(nb.CLIENT_LABELS.get(cid, cid)),
            "rows": int(len(df)),
            "cols": int(df.shape[1]),
            "quality_score": round(q, 3),
        }
    )
quality_df = pd.DataFrame(rows).sort_values("client").reset_index(drop=True)
quality_df


In [ ]:
# Figure 1: quality scores and row counts by country
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(quality_df["country"], quality_df["quality_score"], color=["#2a6f97","#d1495b","#edae49","#00798c"])
axes[0].set_ylim(0, 105); axes[0].set_ylabel("Score (0-100)"); axes[0].set_title("Overall data quality score")
axes[1].bar(quality_df["country"], quality_df["rows"], color="#6a4c93")
axes[1].set_ylabel("Rows (loaded)"); axes[1].set_title(f"Rows per client (mode={nb.MODE})")
for ax in axes: ax.tick_params(axis='x', rotation=20)
fig.tight_layout(); nb.save_fig(fig, "synthetic_data_quality_bars"); plt.show()


**How to read this / what it shows.** Left: per-client tabular quality (higher is better) from `DataQualityAnalyzer`. Right: number of rows analyzed in this run. Similar scores indicate consistent generation hygiene; differences may reflect sampling or country-specific distributions/outliers.


## Cross-country distributions and shift
We visualize key numeric features by country. Because no verified external "real" dataset is provided here, this is a **client-vs-client** (synthetic heterogeneity) comparison, not a real-vs-synthetic fidelity test.


In [ ]:
# Figure 2: histograms/KDE-like density via hist for key features
fig, axes = plt.subplots(len(key_feats), 1, figsize=(8, 2.6*len(key_feats)), sharex=False)
if len(key_feats) == 1: axes = [axes]
for ax, feat in zip(axes, key_feats):
    for c, df in clients.items():
        vals = pd.to_numeric(df[feat], errors='coerce').dropna().values
        ax.hist(vals, bins=30, alpha=0.35, density=True, label=nb.CLIENT_LABELS.get(c,c))
    ax.set_title(f"Distribution: {feat}"); ax.set_xlabel(feat); ax.set_ylabel("Density")
axes[0].legend(ncol=2)
fig.tight_layout(); nb.save_fig(fig, "synthetic_data_hist_grid"); plt.show()


**How to read this / what it shows.** Overlaid densities per country for selected features. Separation between countries indicates covariate/target distribution shift (non-IID data), which affects federated optimization and transfer.


In [ ]:
# Figure 3: ECDFs of key features
def ecdf(x):
    x = np.sort(np.asarray(x)); y = np.arange(1, len(x)+1)/len(x); return x, y
fig, axes = plt.subplots(1, len(key_feats), figsize=(4*len(key_feats), 3.8))
if len(key_feats) == 1: axes = [axes]
for ax, feat in zip(axes, key_feats):
    for c, df in clients.items():
        vals = pd.to_numeric(df[feat], errors='coerce').dropna().values
        x,y = ecdf(vals); ax.plot(x,y,label=nb.CLIENT_LABELS.get(c,c))
    ax.set_title(f"ECDF: {feat}"); ax.set_xlabel(feat); ax.set_ylabel("F(x)")
axes[0].legend(fontsize=8)
fig.tight_layout(); nb.save_fig(fig, "synthetic_data_ecdf"); plt.show()


**How to read this / what it shows.** ECDFs show full distributional differences (location/scale/tails) without binning choices. Horizontal gaps at a fixed y indicate quantile shifts.


In [ ]:
# Figure 4: correlation heatmaps per country + absolute difference to first
corrs = {}
use_cols = [c for c in num_cols if c in key_feats or c in ['age','annual_income','savings_balance','credit_score','debt_to_income_ratio','number_of_claims','claim_risk_score','customer_lifetime_value','premium_amount']][:10]
for c, df in clients.items():
    corrs[c] = df[use_cols].corr(numeric_only=True)
countries = list(clients.keys())
fig, axes = plt.subplots(1, len(countries)+1, figsize=(4*(len(countries)+1), 3.8))
for ax, c in zip(axes[:-1], countries):
    im = ax.imshow(corrs[c].values, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_title(nb.CLIENT_LABELS.get(c,c)); ax.set_xticks(range(len(use_cols))); ax.set_yticks(range(len(use_cols)))
    ax.set_xticklabels(use_cols, rotation=90, fontsize=6); ax.set_yticklabels(use_cols, fontsize=6)
diff = (corrs[countries[0]] - corrs[countries[1]]).abs() if len(countries)>1 else corrs[countries[0]]*0
axes[-1].imshow(diff.values, vmin=0, vmax=1, cmap='magma')
axes[-1].set_title(f"|corr diff| {nb.CLIENT_LABELS.get(countries[0],'A')} vs {nb.CLIENT_LABELS.get(countries[1],'B')}")
axes[-1].set_xticks(range(len(use_cols))); axes[-1].set_yticks(range(len(use_cols)))
axes[-1].set_xticklabels(use_cols, rotation=90, fontsize=6); axes[-1].set_yticklabels(use_cols, fontsize=6)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='correlation')
fig.tight_layout(); nb.save_fig(fig, "synthetic_data_corr_heatmaps"); plt.show()


**How to read this / what it shows.** Per-country Pearson correlations (left panels) and an absolute-difference heatmap (right). Bright difference cells indicate feature relationships that change across countries.


In [ ]:
# Figure 5: PCA 2D scatter by country + missingness
from sklearn.impute import SimpleImputer
sample_frames = []
for c, df in clients.items():
    s = df[num_cols].sample(min(cfg.n_pca//4, len(df)), random_state=nb.SEED).copy(); s['client']=str(c); sample_frames.append(s)
pca_df = pd.concat(sample_frames, ignore_index=True)
pca_df['client'] = pca_df['client'].astype(str)
X = SimpleImputer(strategy='median').fit_transform(pca_df[num_cols].values)
X = StandardScaler().fit_transform(X)
Z = PCA(n_components=2, random_state=nb.SEED).fit_transform(X)
fig, axes = plt.subplots(1,2, figsize=(11,4.2))
for c in countries:
    m = (pca_df['client'].astype(str)==str(c)).values
    axes[0].scatter(Z[m,0], Z[m,1], s=10, alpha=0.5, label=str(nb.CLIENT_LABELS.get(c,c)))
axes[0].set_title('PCA (2D) of sampled numeric features'); axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].legend(markerscale=2)
miss = pd.DataFrame({str(nb.CLIENT_LABELS.get(c,c)): df[num_cols].isna().mean().values for c,df in clients.items()}, index=num_cols)
miss.mean(axis=1).sort_values(ascending=False).head(12).plot(kind='barh', ax=axes[1], color='#00798c')
axes[1].set_title('Mean missingness rate (top 12 numeric cols)'); axes[1].set_xlabel('Fraction missing')
fig.tight_layout(); nb.save_fig(fig, "synthetic_data_pca_missingness"); plt.show()


**How to read this / what it shows.** Left: 2D PCA projection colored by country; overlap suggests similar global structure, clusters/separation indicate heterogeneity. Right: average missingness; near-zero bars indicate complete synthetic fields (expected for generated data).


## Utility test: train-on-A / test-on-B (TSTR-style)
We predict `is_high_risk` (binary) with a standardized logistic regression. Each train/test split is disjoint within the training country; evaluation is on another country's loaded sample.


In [ ]:
countries = [str(c) for c in nb.CLIENTS if str(c) in {str(k) for k in clients.keys()}]
target = 'is_high_risk' if 'is_high_risk' in sample_df.columns else None
auc = pd.DataFrame(index=countries, columns=countries, dtype=float)
if target is None:
    print('[fallback] is_high_risk not available; skipping TSTR-style utility matrix')
else:
    feat_cols = [c for c in num_cols if c != target]
    for a in countries:
        dfa = clients[a][feat_cols + [target]].dropna().copy()
        if len(dfa) < 20 or dfa[target].nunique() < 2:
            auc.loc[a, :] = np.nan
            continue
        try:
            Xtr, _, ytr, _ = train_test_split(
                dfa[feat_cols], dfa[target],
                test_size=0.3, random_state=nb.SEED,
                stratify=dfa[target] if dfa[target].nunique() > 1 else None,
            )
        except Exception:
            Xtr, _, ytr, _ = train_test_split(dfa[feat_cols], dfa[target], test_size=0.3, random_state=nb.SEED)
        model = Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc', StandardScaler()),
            ('lr', LogisticRegression(max_iter=300, random_state=nb.SEED)),
        ])
        if ytr.nunique() < 2:
            auc.loc[a, :] = np.nan
            continue
        model.fit(Xtr, ytr)
        for b in countries:
            dfb = clients[b][feat_cols + [target]].dropna().copy()
            if len(dfb) < 20 or dfb[target].nunique() < 2:
                auc.loc[a, b] = np.nan
                continue
            try:
                _, Xte, _, yte = train_test_split(
                    dfb[feat_cols], dfb[target],
                    test_size=0.5, random_state=nb.SEED,
                    stratify=dfb[target] if dfb[target].nunique() > 1 else None,
                )
            except Exception:
                _, Xte, _, yte = train_test_split(dfb[feat_cols], dfb[target], test_size=0.5, random_state=nb.SEED)
            if yte.nunique() < 2:
                auc.loc[a, b] = np.nan
            else:
                try:
                    auc.loc[a, b] = float(roc_auc_score(yte, model.predict_proba(Xte)[:, 1]))
                except Exception:
                    auc.loc[a, b] = np.nan
auc


In [ ]:
# Figure 6: TSTR-style AUC matrix heatmap
fig, ax = plt.subplots(figsize=(6.2,4.8))
mat = auc.astype(float).values
im = ax.imshow(mat, vmin=0.5, vmax=1.0, cmap='viridis')
ax.set_xticks(range(len(countries))); ax.set_yticks(range(len(countries)))
labels = [nb.CLIENT_LABELS.get(c,c) for c in countries]
ax.set_xticklabels(labels, rotation=30); ax.set_yticklabels(labels)
ax.set_xlabel('Test country'); ax.set_ylabel('Train country'); ax.set_title('Train→Test AUC (is_high_risk)')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i,j]
        if np.isfinite(v): ax.text(j,i,f"{v:.2f}", ha='center', va='center', color='white', fontsize=9)
fig.colorbar(im, ax=ax, label='AUC')
fig.tight_layout(); nb.save_fig(fig, "synthetic_data_tstr_auc"); plt.show()


**How to read this / what it shows.** Rows are training countries, columns test countries. Diagonal values reflect in-domain utility; off-diagonal drops quantify transfer loss under distribution shift.


## Privacy check: distance-to-closest-record (DCR) and duplicates
We compute nearest-neighbor distances between disjoint train/val samples (per client, standardized). Small distances/duplicates can indicate memorization risk in synthetic outputs. **This is not a privacy guarantee.**


In [ ]:
dcr_rows = []
rng_local = np.random.default_rng(nb.SEED)
for c in countries:
    df = clients[c]
    cols = [x for x in num_cols if x != target] if target else num_cols
    data = df[cols].dropna().values.astype(float)
    if len(data) < 50:
        dcr_rows.append({
            "client": str(c),
            "country": str(nb.CLIENT_LABELS.get(c, c)),
            "n_pairs": 0,
            "dcr_mean": np.nan,
            "dcr_p05": np.nan,
            "dcr_min": np.nan,
            "dup_frac": np.nan,
        })
        continue
    sc = StandardScaler().fit(data)
    Z = sc.transform(data)
    n = min(cfg.n_nn, len(Z))
    if n < 4:
        continue
    idx = rng_local.choice(len(Z), size=n, replace=False)
    split = max(1, n // 2)
    Za = Z[idx[:split]]
    Zb = Z[idx[split:]]
    if len(Za) == 0 or len(Zb) == 0:
        continue
    nn = NearestNeighbors(n_neighbors=1).fit(Za)
    d = nn.kneighbors(Zb)[0].ravel()
    dup = float(np.mean(np.isclose(d, 0.0, atol=1e-12)))
    dcr_rows.append({
        "client": str(c),
        "country": str(nb.CLIENT_LABELS.get(c, c)),
        "n_pairs": int(len(d)),
        "dcr_mean": float(np.mean(d)),
        "dcr_p05": float(np.quantile(d, 0.05)),
        "dcr_min": float(np.min(d)),
        "dup_frac": dup,
    })
dcr_df = pd.DataFrame(dcr_rows)
if not dcr_df.empty:
    assert (dcr_df['n_pairs'].fillna(0) >= 0).all()
    if dcr_df['dcr_mean'].notna().any():
        assert (dcr_df['dcr_mean'].dropna() >= 0).all()
    if dcr_df['dup_frac'].notna().any():
        assert ((dcr_df['dup_frac'].dropna() >= 0) & (dcr_df['dup_frac'].dropna() <= 1)).all()
dcr_df


In [ ]:
# Figure 7: DCR histograms and summary bars
fig, axes = plt.subplots(1,2, figsize=(11,4))
for c, df in clients.items():
    cols = [x for x in num_cols if x != target] if target else num_cols
    data = df[cols].dropna().values.astype(float)
    if len(data) < 50: continue
    sc = StandardScaler().fit(data); Z = sc.transform(data)
    n = min(cfg.n_nn, len(Z)); idx = np.random.default_rng(nb.SEED).choice(len(Z), size=n, replace=False)
    Za, Zb = Z[idx[:n//2]], Z[idx[n//2:]]
    d = NearestNeighbors(n_neighbors=1).fit(Za).kneighbors(Zb)[0].ravel()
    axes[0].hist(d, bins=30, alpha=0.35, density=True, label=nb.CLIENT_LABELS.get(c,c))
axes[0].set_title('DCR distribution (train/val split, standardized)'); axes[0].set_xlabel('Distance to closest record'); axes[0].set_ylabel('Density'); axes[0].legend()
if not dcr_df.empty:
    axes[1].bar(dcr_df['country'], dcr_df['dcr_mean'], yerr=None, color='#2a6f97', label='mean DCR')
    ax2 = axes[1].twinx(); ax2.plot(dcr_df['country'], dcr_df['dup_frac'], 'o-', color='#d1495b', label='dup frac')
    axes[1].set_ylabel('Mean DCR'); ax2.set_ylabel('Exact/near-zero fraction'); axes[1].set_title('DCR summary by country')
fig.tight_layout(); nb.save_fig(fig, "synthetic_data_dcr"); plt.show()


**How to read this / what it shows.** Left: distribution of nearest-neighbor distances between disjoint samples; mass near zero would indicate near-duplicates. Right: mean DCR and near-zero fraction per country. Values depend on sample size and feature scaling.


## Optional: t-SNE (small sample)
Run only if `scikit-learn` provides t-SNE and sample is small (light mode caps size) to limit runtime.


In [ ]:
try:
    from sklearn.manifold import TSNE
    sample_frames = []
    for cid in countries:
        df = clients[cid]
        s = df[num_cols].sample(min(cfg.n_tsne // 4, len(df)), random_state=nb.SEED).copy()
        s['client'] = str(cid)
        sample_frames.append(s)
    tdf = pd.concat(sample_frames, ignore_index=True)
    tdf['client'] = tdf['client'].astype(str)
    X = StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(tdf[num_cols].values))
    Xs = X[:min(cfg.n_tsne, len(X))]
    labs = np.asarray(tdf['client'].astype(str).values[:len(Xs)])
    emb = TSNE(
        n_components=2,
        perplexity=min(30, max(5, len(Xs) // 20)),
        random_state=nb.SEED,
        init='pca',
        learning_rate='auto',
    ).fit_transform(Xs)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for cid in countries:
        m = labs.astype(str) == str(cid)
        ax.scatter(emb[m, 0], emb[m, 1], s=12, alpha=0.6, label=nb.CLIENT_LABELS.get(cid, cid))
    ax.set_title('t-SNE of sampled records (illustrative)')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.legend()
    fig.tight_layout()
    nb.save_fig(fig, "synthetic_data_tsne")
    plt.show()
except Exception as e:
    print('[fallback] t-SNE unavailable or failed:', e)


**How to read this / what it shows.** t-SNE preserves local neighborhoods in the embedding; it is sensitive to perplexity/sampling and should be read qualitatively, not as a distance-preserving map.


## Seed stability (synthetic-vs-synthetic proxy)
Without a verified external real dataset, stability is checked by resampling with a different seed and comparing summary statistics (illustrative of generator/sampling variability in this analysis).


In [ ]:
# Figure 8: seed stability of key feature means
alt = {}
for c in countries:
    df = clients[c]
    alt[c] = df.sample(min(len(df), cfg.n_rows), random_state=nb.SEED + 1) if len(df) > 10 else df.copy()
rows = []
for feat in key_feats:
    for c in countries:
        a = pd.to_numeric(clients[c][feat], errors='coerce').mean()
        b = pd.to_numeric(alt[c][feat], errors='coerce').mean()
        rows.append({
            'feature': feat,
            'country': str(nb.CLIENT_LABELS.get(c, c)),
            'mean_seed42': float(a) if pd.notna(a) else np.nan,
            'mean_seed43': float(b) if pd.notna(b) else np.nan,
            'abs_diff': float(abs(a - b)) if pd.notna(a) and pd.notna(b) else np.nan,
        })
stab = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 4))
for feat in key_feats:
    sub = stab[stab.feature == feat]
    ax.plot(sub['country'], sub['abs_diff'], marker='o', label=feat)
ax.set_title('Seed resample stability: |mean diff| (seed 42 vs 43 sample)')
ax.set_ylabel('Absolute mean difference')
ax.set_xlabel('Country')
ax.legend()
fig.tight_layout()
nb.save_fig(fig, "synthetic_data_seed_stability")
plt.show()
stab.head()


**How to read this / what it shows.** Small differences indicate that reported summaries are stable under resampling in this run; large differences would suggest high variance or small effective samples.


## Quantitative summary table


In [ ]:
summary = quality_df.copy()
summary['client'] = summary['client'].astype(str)
if not dcr_df.empty:
    dcr_merge = dcr_df[['client', 'dcr_mean', 'dcr_p05', 'dup_frac']].copy()
    dcr_merge['client'] = dcr_merge['client'].astype(str)
    summary = summary.merge(dcr_merge, on='client', how='left')
if target is not None and not auc.empty:
    diag = pd.Series({str(c): auc.loc[c, c] for c in countries}, name='in_domain_auc', dtype=float)
    diag_df = diag.rename_axis('client').reset_index()
    diag_df['client'] = diag_df['client'].astype(str)
    summary = summary.merge(diag_df, on='client', how='left')
summary


## Sensitivity / ablation
- **Sample size:** DCR and AUC depend on `cfg.n_rows`, `cfg.n_nn`; rerun with `NB_MODE=full` for tighter estimates.
- **Feature set/scaling:** DCR uses standardized numeric features; adding/removing features changes nearest-neighbor geometry.
- **Task choice:** utility matrix uses `is_high_risk`; other targets/tasks may show different transfer patterns.


## Limitations and failure cases
- No verified real dataset is compared here; fidelity claims to real-world distributions are out of scope.
- Correlation/utility analyses are associative, not causal.
- AUC can be unstable with class imbalance or small splits; NaNs indicate degenerate label splits.
- DCR/duplicate checks do not detect all privacy risks (e.g., attribute inference, membership inference).


## Conclusions and further experiments
- The notebook computes **real per-client metrics** from bundled data: quality scores, distribution/correlation diagnostics, transfer AUC, and DCR summaries.
- Cross-country differences (when present) motivate federated or personalized modeling.
- Further work: compare against a verified holdout/real source (if available), add classifier two-sample tests, membership inference evaluation, and formal DP accounting for generation/training pipelines.


## References
- K. El Emam, L. Mosquera, and R. Hoptroff. *Practical Synthetic Data Generation*. O'Reilly, 2020. (general background)
- J. Yoon, L. N. Drumright, and M. van der Schaar. "Anonymization Through Data Synthesis Using Generative Adversarial Networks (ADS-GAN)." *IEEE J. Biomed. Health Inform.*, 2020. (privacy/utility context)
- Scikit-learn developers. *scikit-learn: Machine Learning in Python*, https://scikit-learn.org/ (PCA, t-SNE, metrics, preprocessing APIs used).
